# 🎨 실시간 AI 비주얼 — 음악에 반응하는 AI 이미지

피아노 음악을 재생하면서 AI가 **실시간으로** 이미지를 생성합니다.
음량이 커지면 이미지가 격렬하게 변하고, 조용해지면 부드럽게 흐릅니다.

**도구**: SD-Turbo (초경량 1-step 이미지 생성) + TAESD (초소형 디코더)

- 프레임당 약 0.2초 — **초당 3~5장** 생성
- VRAM 약 3.5GB — T4에서 여유롭게 실행

▶ 먼저 아래 셀을 실행해주세요. 설치에 2~3분 정도 걸립니다.

In [ ]:
%%capture
!pip install -q diffusers transformers accelerate librosa soundfile matplotlib Pillow

import warnings
warnings.filterwarnings('ignore')

print('✅ 설치 완료!')

---
## 1. 피아노 연주 파일 올리기

▶ 영상을 만들 피아노 연주 파일을 업로드하세요.
파일이 없으면 다음 셀에서 예시를 사용하세요.

In [ ]:
from google.colab import files
import os

uploaded = files.upload()
input_audio = list(uploaded.keys())[0]
print(f'✅ 파일이 업로드되었습니다: {input_audio}')

---
## 2. (선택) 예시 파일 사용

▶ 파일이 없으면 이 셀을 실행하세요.

In [ ]:
import urllib.request
import IPython.display as ipd

REPO_URL = 'https://raw.githubusercontent.com/joonhyungbae/pianokit/main/assets'
input_audio = 'piano_chopin.wav'

if not os.path.exists(input_audio):
    try:
        urllib.request.urlretrieve(f'{REPO_URL}/{input_audio}', input_audio)
        print(f'✅ 다운로드 완료: {input_audio}')
    except:
        print(f'⚠️ 다운로드 실패. 위 셀에서 직접 업로드해주세요.')

print(f'\n🎵 원본 오디오:')
if os.path.exists(input_audio):
    ipd.display(ipd.Audio(input_audio))

---
## 3. 오디오 분석

음악의 음량(RMS)과 비트를 미리 분석합니다.

▶ 아래 셀을 실행하세요.

In [ ]:
import librosa
import numpy as np

y, sr = librosa.load(input_audio, sr=22050)
duration = len(y) / sr

# RMS (음량)
rms = librosa.feature.rms(y=y)[0]
rms_norm = (rms - rms.min()) / (rms.max() - rms.min() + 1e-8)
rms_times = librosa.times_like(rms, sr=sr)

# Beat
tempo, beats = librosa.beat.beat_track(y=y, sr=sr)
beat_times = librosa.frames_to_time(beats, sr=sr)

print(f'🎵 오디오: {duration:.1f}초, 템포 {tempo:.0f} BPM, {len(beat_times)}개 비트')

---
## 4. AI 모델 로드

**SD-Turbo**는 단 1스텝으로 이미지를 생성하는 초경량 모델입니다.
**TAESD**는 초소형 디코더로, 이미지 변환 속도를 더욱 높여줍니다.

⏰ 모델 다운로드에 1~2분 걸립니다. ▶ 아래 셀을 실행하세요.

In [ ]:
import torch
from diffusers import AutoPipelineForText2Image, AutoPipelineForImage2Image, AutoencoderTiny

print('🔄 SD-Turbo + TAESD 모델을 불러오는 중...')

# SD-Turbo: 860M params, ~3.4GB 다운로드
model_id = 'stabilityai/sd-turbo'
# 더 높은 품질을 원하면 아래 모델로 교체 (6.5GB 다운로드, 프레임당 ~800ms):
# model_id = 'stabilityai/sdxl-turbo'

pipe_txt = AutoPipelineForText2Image.from_pretrained(
    model_id, torch_dtype=torch.float16, variant='fp16'
)

# TAESD: 초소형 디코더로 교체 (49M → 1.2M params, 디코딩 ~40배 빠름)
pipe_txt.vae = AutoencoderTiny.from_pretrained(
    'madebyollin/taesd', torch_dtype=torch.float16
).to('cuda')

pipe_txt.to('cuda')

# img2img — 같은 컴포넌트 공유 (추가 VRAM 없음)
pipe_img = AutoPipelineForImage2Image.from_pipe(pipe_txt)

print('✅ 모델 로딩 완료! (VRAM ~3.5GB)')

---
## 5. 프롬프트 설정

▶ 영상의 분위기를 영어로 적어주세요.

| 프롬프트 | 분위기 |
|---------|-------|
| `abstract flowing shapes, dark blue and gold` | 추상 금빛 |
| `cosmic nebula, deep purple and cyan` | 우주 성운 |
| `underwater bioluminescent world` | 심해 발광 |

In [ ]:
prompt = "abstract flowing shapes, dark blue and gold"  # ← 여기를 수정하세요

print(f'🎨 프롬프트: "{prompt}"')

---
## 6. 실시간 생성!

▶ 아래 셀을 실행하면 오디오를 재생하면서 AI가 **실시간으로** 이미지를 생성합니다.

음량이 커지면 이미지가 크게 변하고, 비트마다 장면이 전환됩니다.
노트북 안에서 이미지가 계속 갱신되는 것을 확인하세요.

⏰ 오디오 길이만큼 실행됩니다.

In [ ]:
import time
import matplotlib.pyplot as plt
from IPython.display import display, clear_output, Audio
from PIL import Image

duration_sec = min(duration, 30)  # 최대 30초 (더 길면 잘라냄)
fps_target = 4  # 목표 FPS

total_frames = int(duration_sec * fps_target)
print(f'🎬 {duration_sec:.0f}초, 목표 {fps_target}fps → 약 {total_frames}프레임')

# 비트 프레임 변환
beat_frame_set = set([int(bt * fps_target) for bt in beat_times if bt < duration_sec])

# 오디오 재생 시작
display(Audio(input_audio, autoplay=True))
print('🎵 오디오 재생 시작!\n')

# 첫 프레임 생성
seed = 42
gen = torch.Generator('cuda').manual_seed(seed)

prev_image = pipe_txt(
    prompt=prompt,
    num_inference_steps=1,
    guidance_scale=0.0,
    generator=gen,
    width=512, height=512
).images[0]

# 실시간 루프
fig, ax = plt.subplots(1, 1, figsize=(7, 7))
img_display = ax.imshow(prev_image)
ax.axis('off')
ax.set_title(f'🎨 "{prompt[:40]}..."', fontsize=11)

start_time = time.time()
frame_times = []

for frame_idx in range(1, total_frames):
    t = frame_idx / fps_target

    # 현재 시간의 RMS
    rms_idx = min(int(t / duration * len(rms_norm)), len(rms_norm) - 1)
    current_rms = rms_norm[rms_idx]

    # 음량 → 변화 강도
    strength = 0.3 + current_rms * 0.5

    # 비트 → 장면 전환
    if frame_idx in beat_frame_set:
        seed += 1

    gen = torch.Generator('cuda').manual_seed(seed)

    frame_start = time.time()

    # 1-step img2img 생성
    new_image = pipe_img(
        prompt=prompt,
        image=prev_image,
        strength=min(strength, 0.85),
        guidance_scale=0.0,
        num_inference_steps=1,
        generator=gen,
    ).images[0]

    frame_ms = (time.time() - frame_start) * 1000
    frame_times.append(frame_ms)

    # 실시간 화면 갱신
    img_display.set_data(new_image)
    elapsed = time.time() - start_time
    ax.set_xlabel(f't={t:.1f}s | {frame_ms:.0f}ms/frame | RMS={current_rms:.2f}', fontsize=9)
    fig.canvas.draw()
    clear_output(wait=True)
    display(fig)

    prev_image = new_image

# 결과 요약
avg_ms = np.mean(frame_times)
print(f'\n✅ 완료! {len(frame_times)}프레임 생성')
print(f'⚡ 평균 {avg_ms:.0f}ms/프레임 ({1000/avg_ms:.1f} FPS)')
plt.close(fig)

---
## 7. (선택) 프롬프트 바꿔서 다시 해보기

▶ 위의 프롬프트 셀에서 텍스트를 바꾸고, 6번 셀을 다시 실행해보세요.
같은 음악인데 완전히 다른 비주얼이 나옵니다.

💡 프롬프트 아이디어:
- `neon city lights reflecting on wet streets at night`
- `watercolor flowers blooming in timelapse`
- `ink wash painting, traditional asian landscape`